In [ ]:
import numpy as np
print(np.__version__)

import torch
print(torch.__version__)

import torch.nn.functional as F
from tqdm.notebook import trange

torch.manual_seed(0)

from alphazero.gym_env import GymGame, AtariGym
from alphazero.muzero_atari import MuZeroAtariNetwork, MuZeroVectorNetwork
from alphazero.muzero_gym import MuZeroGym, MuZeroGymMCTS

In [ ]:
# CartPole is a fast sanity check (no Atari ROMs required).
# The same MCTS + training loop works for Atari when you swap the game and model below.

cartpole = GymGame("CartPole-v1", max_episode_steps=200)
obs_dim = cartpole.get_encoded_state(cartpole.get_initial_state()).reshape(-1).shape[0]

model = MuZeroVectorNetwork(cartpole, obs_dim=obs_dim, hidden_dim=64)
model.eval()

args = {
    'C': 1.25,
    'gamma': 0.99,
    'num_searches': 25,
    'dirichlet_epsilon': 0.25,
    'dirichlet_alpha': 0.3,
}

mcts = MuZeroGymMCTS(cartpole, model, args)
state = cartpole.get_initial_state()
action_probs = mcts.search(state)

print("MCTS action probabilities (untrained CartPole):")
print(np.round(action_probs, 3))
print(f"Best action: {np.argmax(action_probs)}")

In [ ]:
cartpole = GymGame("CartPole-v1", max_episode_steps=200)
obs_dim = cartpole.get_encoded_state(cartpole.get_initial_state()).reshape(-1).shape[0]

model = MuZeroVectorNetwork(cartpole, obs_dim=obs_dim, hidden_dim=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

args = {
    'C': 1.25,
    'gamma': 0.99,
    'num_searches': 25,
    'num_iterations': 2,
    'num_selfPlay_iterations': 20,
    'num_epochs': 2,
    'batch_size': 16,
    'num_unroll_steps': 5,
    'dirichlet_epsilon': 0.25,
    'dirichlet_alpha': 0.3,
}

muzero = MuZeroGym(model, optimizer, cartpole, args)

for iteration in range(args['num_iterations']):
    replay_buffer = []
    model.eval()
    returns = []
    for _ in trange(args['num_selfPlay_iterations'], desc=f"Self-play {iteration}"):
        game = muzero.self_play()
        replay_buffer.append(game)
        returns.append(game['episode_return'])
    model.train()
    for _ in trange(args['num_epochs'], desc=f"Train {iteration}"):
        muzero.train(replay_buffer)
    print(f"Iteration {iteration}: mean return = {np.mean(returns):.2f}")
    torch.save(model.state_dict(), f"muzero_cartpole_{iteration}.pt")

In [ ]:
# Atari training — same MuZero loop, visual encoder + latent dynamics.
# pip install "gymnasium[atari,accept-rom-license]" opencv-python

try:
    game = AtariGym("ALE/Pong-v5", frame_stack=4, frame_skip=4)
    in_channels = game.frame_stack

    model = MuZeroAtariNetwork(game, in_channels=in_channels, num_res_blocks=2, num_hidden=32)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)

    atari_args = {
        'C': 1.25,
        'gamma': 0.99,
        'num_searches': 50,
        'num_iterations': 2,
        'num_selfPlay_iterations': 10,
        'num_epochs': 2,
        'batch_size': 8,
        'num_unroll_steps': 5,
        'dirichlet_epsilon': 0.25,
        'dirichlet_alpha': 0.3,
    }

    muzero_atari = MuZeroGym(model, optimizer, game, atari_args)
    muzero_atari.learn()
except Exception as exc:
    print("Atari training skipped:", exc)

In [ ]:
cartpole = GymGame("CartPole-v1", max_episode_steps=200)
obs_dim = cartpole.get_encoded_state(cartpole.get_initial_state()).reshape(-1).shape[0]

model = MuZeroVectorNetwork(cartpole, obs_dim=obs_dim, hidden_dim=64)
model.eval()

args = {'C': 1.25, 'gamma': 0.99, 'num_searches': 50, 'dirichlet_epsilon': 0., 'dirichlet_alpha': 0.3}
mcts = MuZeroGymMCTS(cartpole, model, args)

state = cartpole.get_initial_state()
total_reward = 0

while True:
    action_probs = mcts.search(state)
    action = int(np.argmax(action_probs))
    state = cartpole.get_next_state(state, action)
    reward, done = cartpole.get_value_and_terminated(state, action)
    total_reward += reward
    if done:
        print(f"Episode finished. Return = {total_reward}")
        break